# Standard formats for all datasets involved

- Year: `Year`
- Origin: `Origin ISO`, using ISO3 codes
- Destination: `Destination ISO`, using ISO3 codes

In [7]:
import pandas as pd
import xarray as xr
import numpy as np

## Conversion table

In [8]:
# Import all the code conversion tables
code_to_iso3 = (pd.read_csv("/Users/thomasgaskin/UN_migration_data/Iso_code_lookup.csv")[["Numeric", "Alpha-3 code"]]).set_index("Numeric")
code_to_iso3 = code_to_iso3[~code_to_iso3.index.isna()]
code_to_iso3.index = code_to_iso3.index.astype(int)
code_to_iso3 = code_to_iso3.to_dict()['Alpha-3 code']
iso3_to_name = pd.read_csv("/Users/thomasgaskin/UN_migration_data/Iso_code_lookup.csv").set_index('Alpha-3 code')['Country'].to_dict()
iso2_to_3 = (pd.read_csv("/Users/thomasgaskin/UN_migration_data/Iso_code_lookup.csv")[["Alpha-2 code", "Alpha-3 code"]]).set_index("Alpha-2 code").to_dict()['Alpha-3 code']

## UN Net migration data

In [3]:
# Import UN net migration data
UN_data = pd.read_csv("/Users/thomasgaskin/UN_migration_data/UN_data.csv", low_memory=False, index_col=0).drop(["Notes", "Location code", "SDMX code**", "Parent code", "ISO2 Alpha-code"], axis=1).set_index(["Variant", "Year", "ISO3 Alpha-code"])

# Only select country category and convert to xarray
UN_data = UN_data[UN_data.Type == "Country/Area"].to_xarray().squeeze(drop=True)

# Rename coordinates
UN_data = UN_data.assign_coords({"Year": UN_data.coords["Year"].data.astype(int)}).rename({'ISO3 Alpha-code': 'Country ISO'})

# Drop the Holy See (all NAN)
UN_data = UN_data.drop_sel({"Country ISO": "VAT"})

# Convert strings to floats
for var in UN_data.data_vars:
    try:
        UN_data[var].data = np.reshape([float(s.replace(' ', '')) for s in UN_data[var].data.flatten()], UN_data[var].shape)
    except:
        continue

In [7]:
# Save to netcdf
UN_data.to_netcdf("../../data/Migration/UN_net_migration.nc")

## UN Stock data

In [8]:
# Import the UN Stock data and adjust
stock_data = pd.read_csv("/Users/thomasgaskin/UN_migration_data/UN_stock_data/Table 1-Table 1.csv", low_memory=False, index_col=0)
stock_data.drop(["1990.1", "1995.1", "2000.1", "2005.1", "2010.1", "2015.1", "2020.1", 
                 "1990.2", "1995.2", "2000.2", "2005.2", "2010.2", "2015.2", "2020.2", 
                 "Notes of destination", "Type of data of destination"], axis=1, inplace=True)
stock_data['Origin ISO'] = [code_to_iso3.get(x, None) for x in stock_data['Location code of origin']]
stock_data['Destination ISO'] = [code_to_iso3.get(x, None) for x in stock_data['Location code of destination']]

rows_to_drop = []

# Drop flows that are not between countries
for idx, row in stock_data.iterrows():
    if row['Origin ISO'] is None or row['Destination ISO'] is None:
        rows_to_drop.append(idx)
stock_data.drop(rows_to_drop, axis=0, inplace=True)
print(f'Dropped {len(rows_to_drop)} rows')

# Reshape and convert to numbers
stock_data.rename({'Location code of origin': 'Origin Numeric', 
                   'Location code of destination': 'Destination Numeric',
                   'Region, development group, country or area of destination': 'Destination', 
                   'Region, development group, country or area of origin': 'Origin'}, axis=1, inplace=True)
def replace(_s):
    if _s == "0..":
        return np.nan 
    else:
        return(int(_s.replace(' ', '')))
for col in ["1990", "1995", "2000", "2005", "2010", "2015", "2020"]:
    stock_data[col] = [replace(s) for s in stock_data[col]]

# Make the year a coordinate
l = []
for year in ["1990", "1995", "2000", "2005", "2010", "2015", "2020"]:
    data_year = stock_data[["Origin ISO", "Destination ISO", year]].copy()
    data_year["Year"] = int(year)
    data_year.rename({year: "Stock"}, axis=1, inplace=True)
    l.append(data_year)

# Reshape and convert to xarray
stock_data = pd.concat(l, axis=0).set_index(["Origin ISO", "Destination ISO", "Year"])
stock_data_xr = stock_data.to_xarray()['Stock']

# Drop the Holy See, since there is no inflow 
stock_data_xr = stock_data_xr.drop_sel({"Origin ISO": "VAT"})

Dropped 25100 rows


In [9]:
# Save to netcdf
stock_data_xr.to_netcdf("../../data/Migration/UN_stock_data.nc")

## Facebook data

In [4]:
# Get the facebook data
fb_data = pd.read_csv("/Users/thomasgaskin/UN_migration_data/international_migration_flow-BETA VERSION 20230730-Data for Good at Meta.csv", low_memory=False)
fb_data["migration_month"] = pd.to_datetime(fb_data["migration_month"])
fb_data["year"] = [x.year for x in fb_data["migration_month"]]
fb_data["month"] = [x.month for x in fb_data["migration_month"]]
fb_data.drop("migration_month", axis=1, inplace=True)
fb_data = fb_data.set_index(["country_from", "country_to", "year", "month"])
fb_data = fb_data.to_xarray().sum('month')
fb_data = fb_data.assign_coords({"country_from": [iso2_to_3[x] for x in fb_data.coords["country_from"].data]})
fb_data = fb_data.assign_coords({"country_to": [iso2_to_3[x] for x in fb_data.coords["country_to"].data]})
fb_data = fb_data.rename({"country_from": "Origin ISO", "country_to": "Destination ISO", "year": "Year"})
fb_data = fb_data['num_migrants']
fb_data = fb_data.where(fb_data > 0, np.nan)

In [6]:
# Save to netcdf
fb_data.to_netcdf("../../data/Migration/Facebook_data.nc")

## New Zealand data

In [12]:
# NZL data
NZ_data = pd.read_csv("/Users/thomasgaskin/UN_migration_data/bilat_nzl.csv").drop("country", axis=1)
NZ_data = NZ_data[[isinstance(s, str) for s in NZ_data.iso3c]]
NZ_data["year"] = NZ_data["year"].astype(int)
NZ_data = NZ_data.set_index(["year", "iso3c", "direction"])
NZ_data = NZ_data[~NZ_data.index.duplicated()].to_xarray()
NZ_data = xr.Dataset({"arrivals": NZ_data["flow"].sel({"direction": "arrivals"}, drop=True),
            "departures": NZ_data["flow"].sel({"direction": "departures"}, drop=True)})
NZ_data = NZ_data.rename({"year": "Year", "iso3c": "Country ISO"})

In [14]:
NZ_data.to_netcdf("../../data/Migration/NZL_data.nc")

## Sweden data

In [17]:
# SWE data
SWE_data = pd.read_csv("/Users/thomasgaskin/UN_migration_data/bilat_swe.csv").set_index(["year", "iso3c", "age", "sex"]).drop("country", axis=1)
SWE_data = SWE_data[~SWE_data.index.duplicated()].to_xarray().sel({"age": "total", "sex": "total"}, drop=True).rename({"imm": "arrivals", "emi": "departures"})
SWE_data = SWE_data.rename({"year": "Year", "iso3c": "Country ISO"})

In [19]:
SWE_data.to_netcdf("../../data/Migration/SWE_data.nc")

## Five-year bilateral estimates

In [25]:
# Five year estimates
five_year_estimates = pd.read_csv("/Users/thomasgaskin/UN_migration_data/bilat_mig.csv").set_index(["year0", "orig", "dest"]).to_xarray()
five_year_estimates = five_year_estimates.rename({"year0": "Year0", "orig": "Origin ISO", "dest": "Destination ISO"})

In [27]:
five_year_estimates.to_netcdf("../../data/Migration/bilateral_estimates.nc")